### Calculating the QM solution of the reaction between $OH^-$ and $CH_3Cl$

In [37]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyscf import gto, scf
from pyscf.geomopt.geometric_solver import optimize
from pyscf.qmmm import mm_charge
from scipy.optimize import minimize
from scipy.spatial.transform import Rotation
import pickle
from pyscf.grad import rhf as rhf_grad

In [38]:
au_to_kJ_conversion = 2625.49962
R = 8.314462618  # J/(mol K)
SCF_CONV_TOL = 1e-6

In [83]:
def qm_water_lj_energy(qm_coords_A, water_O_coords_A, qm_atom_types):
    water_O = np.asarray(water_O_coords_A[0])

    E_LJ = 0.0

    for i, atom_type in enumerate(qm_atom_types):

        # Skip atoms for which we have no LJ parameters
        if atom_type not in qm_lj:
            continue

        sigma = qm_lj[atom_type]["sigma_A"]
        epsilon = qm_lj[atom_type]["epsilon_kJmol"]

        r = np.linalg.norm(
            qm_coords_A[i] - water_O
        )

        E_LJ += lj_energy(
            r,
            sigma,
            epsilon
        )

    return E_LJ

def water_from_variables(x):
    """
    x[:3] = O position in Angstrom
    x[3:6] = rotation vector in radians

    Returns:
        O, H1, H2, M
    """

    translation = np.asarray(x[:3])
    rotation = Rotation.from_rotvec(x[3:6])

    O  = translation + rotation.apply(water_O_local)
    H1 = translation + rotation.apply(water_H1_local)
    H2 = translation + rotation.apply(water_H2_local)
    M  = translation + rotation.apply(water_M_local)

    return O, H1, H2, M

def total_qm_water_energy(x, mol, qm_atom_types, energy_qm):

    # ------------------------------------------------
    # Safety check on water translation
    # ------------------------------------------------

    if np.max(np.abs(x[:3])) > 10.0:
        return 1.0e10

    # Build water
    O, H1, H2, M = water_from_variables(x)

    # ------------------------------------------------
    # TIP4P-D electrostatic sites
    # ------------------------------------------------

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    # ------------------------------------------------
    # QM/MM electrostatics
    # ------------------------------------------------

    mf_qmmm = mm_charge(
        scf.RHF(mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )
    mf_qmmm.conv_tol = SCF_CONV_TOL
    energy_qmmm = mf_qmmm.kernel()

    # ------------------------------------------------
    # QM-water LJ interaction
    # ------------------------------------------------

    E_LJ = qm_water_lj_energy(
        mol.atom_coords(unit="Angstrom"),
        np.array([O]),
        qm_atom_types
    )

    # ------------------------------------------------
    # Electrostatic interaction
    # ------------------------------------------------

    E_electrostatic = (
        energy_qmmm - energy_qm
    ) * au_to_kJ_conversion

    return E_electrostatic + E_LJ

def lj_energy(r_A, sigma_A, epsilon_kJmol):
    """
    Lennard-Jones 12-6 energy.

    Parameters
    ----------
    r_A : float
        Distance in Angstrom
    sigma_A : float
        LJ sigma in Angstrom
    epsilon_kJmol : float
        LJ epsilon in kJ/mol

    Returns
    -------
    float
        LJ energy in kJ/mol
    """
    sr6 = (sigma_A / r_A)**6
    return 4.0 * epsilon_kJmol * (sr6**2 - sr6)

def build_molecule(coords):

    atom_string = ""

    for symbol, coord in zip(symbols, coords):

        x, y, z = coord

        atom_string += (
            f"{symbol} "
            f"{x:.10f} "
            f"{y:.10f} "
            f"{z:.10f}\n"
        )

    return gto.M(
        atom=atom_string,
        basis="6-31G",
        charge=-1,
        spin=0,
        unit="Angstrom"
    )

def move_oh_to_distance(coords, target_distance):

    coords = coords.copy()

    O = 0
    H_oh = 1
    C = 2

    carbon = coords[C]
    oxygen = coords[O]

    # Current direction from C toward O
    direction = oxygen - carbon
    direction /= np.linalg.norm(direction)

    # Preserve the O-H vector
    oh_vector = coords[H_oh] - coords[O]

    # Place O at the desired C-O distance
    new_oxygen = carbon + direction * target_distance

    # Move H together with O
    new_hydrogen = new_oxygen + oh_vector

    coords[O] = new_oxygen
    coords[H_oh] = new_hydrogen

    return coords

def write_constraint(distance):

    with open("constraints.txt", "w") as f:

        f.write("$set\n")
        f.write(
            f"distance 1 3 {distance:.8f}\n"
        )

def optimize_at_distance(mol, distance, maxsteps=25):

    # Create the constraint file
    write_constraint(distance)

    # Build RHF + implicit solvent
    mf = make_scf(mol)

    # Optimize geometry while keeping C-O fixed
    mol_opt = optimize(
        mf,
        constraints="constraints.txt",
        maxsteps=maxsteps
    )

    # Recalculate final energy using the optimized geometry
    mf_final = make_scf(mol_opt)
    energy = mf_final.kernel()

    # Return the optimized geometry and energy
    return mol_opt, energy

def make_scf(mol):

    mf = scf.RHF(mol)
    mf.conv_tol = SCF_CONV_TOL
    return mf

def total_qmmm_lj_energy(coords_A, water_x):

    # Build QM molecule
    mol = build_molecule(coords_A)

    # Build TIP4P-D water
    O, H1, H2, M = water_from_variables(water_x)

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    # ------------------------------------------------
    # Bare QM energy
    # ------------------------------------------------

    mf_qm = scf.RHF(mol)
    mf_qm.conv_tol = SCF_CONV_TOL
    E_qm = mf_qm.kernel()

    # ------------------------------------------------
    # QM/MM energy
    # ------------------------------------------------

    mf_qmmm = mm_charge(
        scf.RHF(mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf_qmmm.conv_tol = SCF_CONV_TOL
    E_qmmm = mf_qmmm.kernel()

    # ------------------------------------------------
    # QM/MM electrostatic interaction
    # ------------------------------------------------

    E_electrostatic = (
        E_qmmm - E_qm
    ) * au_to_kJ_conversion

    # ------------------------------------------------
    # QM-water LJ
    # ------------------------------------------------

    E_LJ = qm_water_lj_energy(
        coords_A,
        np.array([O]),
        qm_atom_types
    )

    # ------------------------------------------------
    # Total interaction energy
    # ------------------------------------------------

    E_total = E_electrostatic + E_LJ

    return E_total
def optimize_qmmm_at_distance(
    mol,
    distance,
    water_x,
    maxiter=25
):

    coords0 = mol.atom_coords(
        unit="Angstrom"
    )

    # ------------------------------------------------
    # C-O distance constraint
    # ------------------------------------------------

    def co_distance(coords_flat):

        coords = coords_flat.reshape((-1, 3))

        C = coords[2]
        O = coords[0]

        return np.linalg.norm(O - C)

    constraint = {
        "type": "eq",
        "fun": lambda x: co_distance(x) - distance
    }

    # ------------------------------------------------
    # QM/MM optimization
    # ------------------------------------------------

    result = minimize(
        lambda x: qmmm_objective(x, water_x),
        coords0.reshape(-1),
        jac=True,
        method="SLSQP",
        constraints=[constraint],
        options={
            "maxiter": maxiter,
            "ftol": 1e-5,
            "disp": True
        }
    )

    # ------------------------------------------------
    # Build optimized molecule
    # ------------------------------------------------

    coords_opt = result.x.reshape((-1, 3))

    mol_opt = build_molecule(coords_opt)

    # ------------------------------------------------
    # Final energies
    # ------------------------------------------------

    mf_qm = make_scf(mol_opt)
    energy_qm = mf_qm.kernel()

    energy_qmmm, _ = qmmm_gradient(
        mol_opt,
        water_x
    )

    return (
        mol_opt,
        energy_qm,
        energy_qmmm,
        result
    )
    
    
def qm_water_lj_energy_gradient(
    qm_coords_A,
    water_O_coords_A,
    qm_atom_types
):
    """
    QM-water LJ energy and gradient.

    Returns
    -------
    E_LJ : float
        LJ energy in kJ/mol

    grad : ndarray, shape (N,3)
        Gradient dE/d(R_QM) in kJ/mol/Angstrom
    """

    water_O = np.asarray(water_O_coords_A[0])

    E_LJ = 0.0
    grad = np.zeros_like(qm_coords_A, dtype=float)

    for i, atom_type in enumerate(qm_atom_types):

        if atom_type not in qm_lj:
            continue

        sigma = qm_lj[atom_type]["sigma_A"]
        epsilon = qm_lj[atom_type]["epsilon_kJmol"]

        # Vector from water O -> QM atom
        dr = qm_coords_A[i] - water_O

        r = np.linalg.norm(dr)

        sr6 = (sigma / r)**6

        # LJ energy
        E_i = 4.0 * epsilon * (sr6**2 - sr6)

        E_LJ += E_i

        # dE/dr
        dE_dr = (
            24.0 * epsilon / r
            * (sr6 - 2.0 * sr6**2)
        )

        # dE/dR_vector
        grad[i] += dE_dr * dr / r

    return E_LJ, grad


class QMMM_LJ_Gradients:

    def __init__(
        self,
        mf,
        water_O_A,
        qm_atom_types,
        mm_coords,
        mm_charges
    ):

        self.mf = mf
        self.mol = mf.mol

        self.water_O_A = np.asarray(water_O_A)
        self.qm_atom_types = qm_atom_types

        self.mm_coords = np.asarray(mm_coords)
        self.mm_charges = np.asarray(mm_charges)

        # Attributes expected by PySCF/geomeTRIC
        self.verbose = mf.verbose
        self.stdout = mf.stdout
        self.converged = True

    def nuc_grad_method(self):
        return self

    def as_scanner(self):
        return self

    def __call__(self, mol):

        # --------------------------------------------
        # QM/MM SCF
        # --------------------------------------------

        mf_qmmm = mm_charge(
            scf.RHF(mol),
            self.mm_coords,
            self.mm_charges,
            unit="Angstrom"
        )

        mf_qmmm.conv_tol = SCF_CONV_TOL

        E_qmmm = mf_qmmm.kernel()

        # --------------------------------------------
        # QM/MM nuclear gradient
        # --------------------------------------------

        grad_qmmm = (
            mf_qmmm
            .nuc_grad_method()
            .kernel()
        )

        # --------------------------------------------
        # LJ energy + gradient
        # --------------------------------------------

        qm_coords_A = mol.atom_coords(
            unit="Angstrom"
        )

        E_LJ, grad_LJ = qm_water_lj_energy_gradient(
            qm_coords_A,
            self.water_O_A,
            self.qm_atom_types
        )

        # --------------------------------------------
        # Convert LJ gradient:
        #
        # kJ/mol/Angstrom
        #        ->
        # Hartree/Bohr
        # --------------------------------------------

        grad_LJ_Ha_Bohr = (
            grad_LJ
            / au_to_kJ_conversion
            / 1.889726125
        )

        # --------------------------------------------
        # Combined energy
        # --------------------------------------------

        E_total = (
            E_qmmm
            + E_LJ / au_to_kJ_conversion
        )

        # --------------------------------------------
        # Combined gradient
        # --------------------------------------------

        grad_total = (
            grad_qmmm
            + grad_LJ_Ha_Bohr
        )

        return E_total, grad_total

def optimize_qmmm_lj_at_distance(
    mol,
    distance,
    water_x,
    maxsteps=25
):

    # ------------------------------------------------
    # 1. Write the C-O constraint
    # ------------------------------------------------

    write_constraint(distance)

    # ------------------------------------------------
    # 2. Build the current TIP4P-D water
    # ------------------------------------------------

    O, H1, H2, M = water_from_variables(water_x)

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    # ------------------------------------------------
    # 3. Create a QM/MM SCF object
    # ------------------------------------------------

    mf = mm_charge(
        scf.RHF(mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf.conv_tol = SCF_CONV_TOL

    # ------------------------------------------------
    # 4. Create our combined QM/MM + LJ gradient
    # ------------------------------------------------

    grad = QMMM_LJ_Gradients(
        mf,
        O,
        qm_atom_types,
        mm_coords,
        mm_charges
    )

    # ------------------------------------------------
    # 5. Geometry optimization
    # ------------------------------------------------

    mol_opt = optimize(
        grad,
        constraints="constraints.txt",
        maxsteps=maxsteps
    )

    # ------------------------------------------------
    # 6. Bare QM energy at optimized geometry
    # ------------------------------------------------

    mf_qm = scf.RHF(mol_opt)
    mf_qm.conv_tol = SCF_CONV_TOL

    energy_qm = mf_qm.kernel()

    # ------------------------------------------------
    # 7. Final QM/MM energy
    # ------------------------------------------------

    mf_qmmm_final = mm_charge(
        scf.RHF(mol_opt),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf_qmmm_final.conv_tol = SCF_CONV_TOL

    energy_qmmm = mf_qmmm_final.kernel()

    # ------------------------------------------------
    # 8. Final LJ energy
    # ------------------------------------------------

    E_LJ = qm_water_lj_energy(
        mol_opt.atom_coords(unit="Angstrom"),
        np.array([O]),
        qm_atom_types
    )

    # ------------------------------------------------
    # 9. Return everything
    # ------------------------------------------------

    return (
        mol_opt,
        energy_qm,
        energy_qmmm,
        E_LJ,
        None
    )

def qmmm_gradient(mol, water_x):

    O, H1, H2, M = water_from_variables(water_x)

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    mf_qmmm = mm_charge(
        scf.RHF(mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf_qmmm.conv_tol = SCF_CONV_TOL

    energy = mf_qmmm.kernel()

    grad = mf_qmmm.nuc_grad_method().kernel()

    return energy, grad

def qmmm_objective(coords_flat, water_x):
    """
    QM/MM energy and gradient for scipy.optimize.

    coords_flat: QM coordinates flattened, in Angstrom
    """

    coords_A = coords_flat.reshape((-1, 3))

    mol = build_molecule(coords_A)

    energy, grad = qmmm_gradient(
        mol,
        water_x
    )

    # Convert gradient from Hartree/Bohr
    # to Hartree/Angstrom because our optimization
    # variables are in Angstrom.
    grad_A = grad / 1.889726125

    return energy, grad_A.reshape(-1)

def qmmm_lj_objective(coords_flat, water_x):
    """
    QM/MM + QM-water LJ energy and gradient.

    coords_flat:
        QM coordinates, flattened, in Angstrom.

    water_x:
        TIP4P-D water position/orientation.

    Returns
    -------
    energy : float
        QM/MM + LJ energy in Hartree.

    gradient : ndarray
        Total gradient in Hartree/Angstrom.
    """

    # ------------------------------------------------
    # Build QM molecule
    # ------------------------------------------------

    coords_A = coords_flat.reshape((-1, 3))

    mol = build_molecule(coords_A)

    # ------------------------------------------------
    # QM/MM energy and gradient
    # ------------------------------------------------

    E_qmmm, grad_qmmm = qmmm_gradient(
        mol,
        water_x
    )

    # grad_qmmm is Hartree/Bohr
    # Convert to Hartree/Angstrom
    grad_qmmm_A = (
        grad_qmmm / 1.889726125
    )

    # ------------------------------------------------
    # TIP4P-D water
    # ------------------------------------------------

    O, H1, H2, M = water_from_variables(water_x)

    # ------------------------------------------------
    # LJ energy and gradient
    # ------------------------------------------------

    E_LJ, grad_LJ = qm_water_lj_energy_gradient(
        coords_A,
        np.array([O]),
        qm_atom_types
    )

    # E_LJ is kJ/mol
    # grad_LJ is kJ/mol/Angstrom

    # Convert LJ energy to Hartree
    E_LJ_Hartree = (
        E_LJ / au_to_kJ_conversion
    )

    # Convert LJ gradient to Hartree/Angstrom
    grad_LJ_Hartree_A = (
        grad_LJ / au_to_kJ_conversion
    )

    # ------------------------------------------------
    # Total
    # ------------------------------------------------

    E_total = (
        E_qmmm
        + E_LJ_Hartree
    )

    grad_total = (
        grad_qmmm_A
        + grad_LJ_Hartree_A
    )

    return E_total, grad_total.reshape(-1)

def qmmm_lj_energy_gradient(coords_flat, water_x):
    """
    QM/MM + LJ energy and gradient for fixed TIP4P-D water.

    Parameters
    ----------
    coords_flat : array, shape (21,)
        QM Cartesian coordinates in Angstrom.
    water_x : array, shape (6,)
        Fixed TIP4P-D water position/orientation.

    Returns
    -------
    energy : float
        Total QM/MM + LJ energy in kJ/mol.

    gradient : array, shape (21,)
        Total gradient in kJ/mol/Angstrom.
    """

    # ------------------------------------------------------------
    # QM coordinates
    # ------------------------------------------------------------

    coords_A = np.asarray(coords_flat).reshape(-1, 3)

    mol = build_molecule(coords_A)

    # ------------------------------------------------------------
    # Fixed TIP4P-D water
    # ------------------------------------------------------------

    O, H1, H2, M = water_from_variables(water_x)

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    # ------------------------------------------------------------
    # QM/MM electrostatics
    # ------------------------------------------------------------

    mf_qmmm = mm_charge(
        scf.RHF(mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf_qmmm.conv_tol = SCF_CONV_TOL

    E_qmmm_Ha = mf_qmmm.kernel()

    grad_qmmm_Ha_Bohr = (
        mf_qmmm.nuc_grad_method().kernel()
    )

    # Convert QM/MM gradient:
    #
    # Hartree/Bohr -> kJ/mol/Angstrom
    #
    grad_qmmm = (
        grad_qmmm_Ha_Bohr
        * au_to_kJ_conversion
        / 1.889726125
    )

    # ------------------------------------------------------------
    # LJ energy + gradient
    # ------------------------------------------------------------

    E_LJ, grad_LJ = qm_water_lj_energy_gradient(
        coords_A,
        np.array([O]),
        qm_atom_types
    )

    # grad_LJ is already kJ/mol/Angstrom
    # based on the gradient function we previously verified.

    # ------------------------------------------------------------
    # Total energy
    # ------------------------------------------------------------

    E_total = (
        E_qmmm_Ha * au_to_kJ_conversion
        + E_LJ
    )

    # ------------------------------------------------------------
    # Total gradient
    # ------------------------------------------------------------

    grad_total = grad_qmmm + grad_LJ

    return E_total, grad_total.ravel()

def optimize_qmmm_lj_at_distance(
    mol,
    distance,
    water_x,
    maxsteps=100
):
    """
    Stage 1:

    - Fixed water
    - Fixed C at origin
    - Fixed O on the -x axis
    - Fixed C-O distance
    - Optimize all other QM coordinates
    """

    coords0 = mol.atom_coords(unit="Angstrom")

    # ------------------------------------------------------------
    # Fixed atoms
    # ------------------------------------------------------------

    C_index = 2
    O_index = 0

    # ------------------------------------------------------------
    # Initial internal geometry
    # ------------------------------------------------------------

    C0 = coords0[C_index].copy()

    # Translate everything so C is at the origin
    coords0 = coords0 - C0

    # Put O at the requested C-O distance
    coords0[O_index] = np.array([
        -distance,
        0.0,
        0.0
    ])

    coords0[C_index] = np.array([
        0.0,
        0.0,
        0.0
    ])

    # ------------------------------------------------------------
    # Variables
    #
    # We optimize atoms 1, 3, 4, 5, 6.
    #
    # O and C are fixed.
    # ------------------------------------------------------------

    variable_atoms = [1, 3, 4, 5, 6]

    x0 = coords0[variable_atoms].ravel()

    # ------------------------------------------------------------
    # Reconstruct full QM geometry
    # ------------------------------------------------------------

    def make_coords(x):

        coords = coords0.copy()

        coords[variable_atoms] = x.reshape(
            len(variable_atoms), 3
        )

        return coords

    # ------------------------------------------------------------
    # Objective
    # ------------------------------------------------------------

    def objective(x):

        coords = make_coords(x)

        E, grad = qmmm_lj_energy_gradient(
            coords.ravel(),
            water_x
        )

        # Only return gradients for variable atoms
        grad = grad.reshape(-1, 3)

        grad_variable = grad[variable_atoms]

        return E, grad_variable.ravel()

    # ------------------------------------------------------------
    # Optimize
    # ------------------------------------------------------------

    result = minimize(
        fun=lambda x: objective(x)[0],
        x0=x0,
        jac=lambda x: objective(x)[1],
        method="BFGS",
        options={
            "maxiter": maxsteps,
            "gtol": 1e-5,
            "disp": True
        }
    )

    # ------------------------------------------------------------
    # Final geometry
    # ------------------------------------------------------------

    coords_opt = make_coords(result.x)

    mol_opt = build_molecule(coords_opt)

    # ------------------------------------------------------------
    # Final bare QM energy
    # ------------------------------------------------------------

    mf_qm = make_scf(mol_opt)
    energy_qm = mf_qm.kernel()

    # ------------------------------------------------------------
    # Final QM/MM energy
    # ------------------------------------------------------------

    O, H1, H2, M = water_from_variables(water_x)

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    mf_qmmm = mm_charge(
        scf.RHF(mol_opt),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf_qmmm.conv_tol = SCF_CONV_TOL

    energy_qmmm = mf_qmmm.kernel()

    # ------------------------------------------------------------
    # Final LJ energy
    # ------------------------------------------------------------

    E_LJ = qm_water_lj_energy(
        coords_opt,
        np.array([O]),
        qm_atom_types
    )

    return (
        mol_opt,
        energy_qm,
        energy_qmmm,
        E_LJ,
        result
    )

In [2]:
qm_lj = {
    "OH_O": {
        "sigma_A": 3.400,
        "epsilon_kJmol": 0.2508914038369354
    },

    "OH_H": {
        "sigma_A": 1.443,
        "epsilon_kJmol": 0.18390926154006562
    },

    "C": {
        "sigma_A": 3.39967,
        "epsilon_kJmol": 0.457730
    },

    "Cl": {
        "sigma_A": 4.04468018036,
        "epsilon_kJmol": 0.6276
    },

    "CH3_H": {
        "sigma_A": 2.64953,
        "epsilon_kJmol": 0.0656888
    }
}


In [4]:
qm_atom_types = [
    "OH_O",
    "OH_H",
    "C",
    "Cl",
    "CH3_H",
    "CH3_H",
    "CH3_H"
]

In [6]:
symbols = [
    "O",
    "H",
    "C",
    "Cl",
    "H",
    "H",
    "H"
]

In [10]:
# Fixed TIP4P-D geometry in a local coordinate system
water_O_local = np.array([0.0, 0.0, 0.0])
water_H1_local = np.array([0.9572, 0.0, 0.0])
water_H2_local = np.array([
    -0.23998617,
     0.92662747,
     0.0
])

water_M_local = np.array([
    0.09462759,
    0.12225716,
    0.0
])

In [11]:
# Initial TIP4P-D water configuration
water_x = np.array([
    2.5, 0.0, 3.0,    # O position
    0.0, 0.0, 0.0     # rotation vector
])

In [22]:
# Testing QM optimization
test_mol = build_molecule(
    np.array([
        [-5.000,  0.000,  0.000],
        [-5.970,  0.000,  0.000],
        [ 0.000,  0.000,  0.000],
        [ 1.780,  0.000,  0.000],
        [-0.630,  0.630,  0.630],
        [-0.630, -0.630,  0.630],
        [-0.630,  0.000, -0.890],
    ])
)
test_distance = 3.0
test_mol_opt, test_energy = optimize_at_distance(
    test_mol,
    test_distance,
    maxsteps=25
)

coords_opt = test_mol_opt.atom_coords(
    unit="Angstrom"
)

actual_distance = np.linalg.norm(
    coords_opt[0] - coords_opt[2]
)

print("\nRESULT")
print("Target C-O distance:", test_distance, "Å")
print("Actual C-O distance:", actual_distance, "Å")
print("Final QM energy:", test_energy, "Hartree")


coords_initial = test_mol.atom_coords(
    unit="Angstrom"
)

print("\nInitial coordinates:")
print(coords_initial)

print("\nOptimized coordinates:")
print(coords_opt)

geometric-optimize called with the following command line:
/home/chemistry/venvs/jupyter/lib/python3.14/site-packages/ipykernel_launcher.py -f /home/chemistry/.local/share/jupyter/runtime/kernel-f483c515-b651-4fa9-ada0-340e3c482655.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%,  **


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -5.000000   0.000000   0.000000    0.000000  0.000000  0.000000
   H  -5.970000   0.000000   0.000000    0.000000  0.000000  0.000000
   C   0.000000   0.000000   0.000000    0.000000  0.000000  0.000000
  Cl   1.780000   0.000000   0.000000    0.000000  0.000000  0.000000
   H  -0.630000   0.630000   0.630000    0.000000  0.000000  0.000000
   H  -0.630000  -0.630000   0.630000    0.000000  0.000000  0.000000
   H  -0.630000   0.000000  -0.890000    0.000000  0.000000  0.000000

WARN: Mole.unit (Angstrom) is changed to Bohr

converged SCF energy = -574.308632234646
--------------- RHF_Scanner gradients ---------------
         x                y                z
0 O    -0.0100830307    -0.0000000000    -0.0003427109
1 H     0.0073938184     0.0000000000     0.0000113669
2 C     0.1146414712    -0.0000000000    -0.0269586810
3 Cl     0.0056602441    -0.0

Step    0 : Gradient = 5.811e-02/1.004e-01 (rms/max) Energy = -574.3086322346
Hessian Eigenvalues: 5.00000e-02 5.00000e-02 5.00000e-02 ... 3.46752e-01 3.47392e-01 5.01282e-01



Geometry optimization cycle 2
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -4.274293  -0.000000   0.000000    0.725707 -0.000000  0.000000
   H  -5.534537   0.000000  -0.000000    0.435463  0.000000 -0.000000
   C  -0.693718  -0.000000   0.000296   -0.693718 -0.000000  0.000296
  Cl   1.202359  -0.000000  -0.046073   -0.577641 -0.000000 -0.046073
   H  -1.153006   0.646901   0.647624   -0.523006  0.016901  0.017624
   H  -1.153006  -0.646901   0.647624   -0.523006 -0.016901  0.017624
   H  -1.215552  -0.000000  -0.879472   -0.585552 -0.000000  0.010528
converged SCF energy = -574.30697958984
--------------- RHF_Scanner gradients ---------------
         x                y                z
0 O     0.0842473412    -0.0000000000    -0.0005904186
1 H    -0.0921802783     0.0000000000    -0.0001401470
2 C    -0.0064944829    -0.0000000000    -0.0238242789
3 Cl     0.0300165832     0.0000000000    -0.0011177672
4 H    -0.0086637070  

Step    1 : Displace = 5.333e-01/9.746e-01 (rms/max) Trust = 1.000e-01 (=) Grad_T = 6.910e-02/9.218e-02 (rms/max) E (change) = -574.3069795898 (+1.653e-03) Quality = 0.003
Constraint                         Current      Target       Diff.
Distance 1-3                       3.58057     3.00000     0.58057
Hessian Eigenvalues: 4.37516e-03 5.00000e-02 5.00000e-02 ... 3.47323e-01 3.98539e-01 5.71866e-01


...


Geometry optimization cycle 25
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -4.094338  -0.000000  -0.599503   -0.001516 -0.000000  0.007136
   H  -4.688585   0.000000   0.168030   -0.005068  0.000000  0.004556
   C  -1.274528  -0.000000   0.416339    0.004358  0.000000 -0.005857
  Cl   0.166942   0.000000  -0.852092   -0.002129  0.000000 -0.011227
   H  -1.104666   0.890372   0.996617    0.006624 -0.000486 -0.005836
   H  -1.104666  -0.890372   0.996617    0.006624  0.000486 -0.005836
   H  -2.260772  -0.000000  -0.108799    0.000983 -0.000000 -0.000477
converged SCF energy = -574.40839545236
--------------- RHF_Scanner gradients ---------------
         x                y                z
0 O    -0.0086189143     0.0000000000    -0.0015298301
1 H    -0.0007707228    -0.0000000000    -0.0010871402
2 C     0.0083580138     0.0000000000     0.0012780306
3 Cl     0.0003522949    -0.0000000000     0.0005071529
4 H     0.0002877284 

Step   24 : Displace = 3.147e-03/7.197e-03 (rms/max) Trust = 3.146e-03 (-) Grad_T = 1.862e-04/3.442e-04 (rms/max) E (change) = -574.4083954524 (+4.088e-05) Quality = 0.964
Constraint                         Current      Target       Diff.
Distance 1-3                       2.99721     3.00000    -0.00279
Hessian Eigenvalues: 4.30029e-03 1.33523e-02 2.35412e-02 ... 3.73762e-01 3.98871e-01 5.00142e-01



Geometry optimization cycle 26
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -4.096367  -0.000000  -0.587201   -0.002029 -0.000000  0.012301
   H  -4.694442   0.000000   0.177470   -0.005857  0.000000  0.009440
   C  -1.269006  -0.000000   0.406662    0.005522 -0.000000 -0.009677
  Cl   0.158925   0.000000  -0.875727   -0.008017  0.000000 -0.023635
   H  -1.093611   0.889813   0.985953    0.011055 -0.000559 -0.010664
   H  -1.093611  -0.889813   0.985953    0.011055  0.000559 -0.010664
   H  -2.260674  -0.000000  -0.108512    0.000098 -0.000000  0.000287
converged SCF energy = -574.408373823936
--------------- RHF_Scanner gradients ---------------
         x                y                z
0 O    -0.0085031505     0.0000000000    -0.0015370714
1 H    -0.0009354191    -0.0000000000    -0.0009959554
2 C     0.0085998560     0.0000000000     0.0010763270
3 Cl     0.0002470561    -0.0000000000     0.0005710614
4 H     0.0002739131

Step   25 : Displace = 4.559e-03/9.464e-03 (rms/max) Trust = 4.449e-03 (+) Grad_T = 2.636e-04/4.754e-04 (rms/max) E (change) = -574.4083738239 (+2.163e-05) Quality = 0.899
Constraint                         Current      Target       Diff.
Distance 1-3                       2.99695     3.00000    -0.00305
Hessian Eigenvalues: 4.30029e-03 1.33523e-02 2.35412e-02 ... 3.73762e-01 3.98871e-01 5.00142e-01
Maximum iterations reached (25); increase --maxiter for more


Geometry optimization failed to converge in 25 iterations
converged SCF energy = -574.408373824827

RESULT
Target C-O distance: 3.0 Å
Actual C-O distance: 2.996954854895541 Å
Final QM energy: -574.4083738248266 Hartree

Initial coordinates:
[[-5.    0.    0.  ]
 [-5.97  0.    0.  ]
 [ 0.    0.    0.  ]
 [ 1.78  0.    0.  ]
 [-0.63  0.63  0.63]
 [-0.63 -0.63  0.63]
 [-0.63  0.   -0.89]]

Optimized coordinates:
[[-4.09636716e+00 -3.32329343e-12 -5.87201439e-01]
 [-4.69444235e+00  3.07047890e-12  1.77470015e-01]
 [-1.26900563e+00 -9.60542844e-13  4.06662308e-01]
 [ 1.58925157e-01  2.94838592e-12 -8.75726834e-01]
 [-1.09361104e+00  8.89812825e-01  9.85952741e-01]
 [-1.09361104e+00 -8.89812825e-01  9.85952741e-01]
 [-2.26067365e+00 -1.11139929e-12 -1.08512397e-01]]


In [21]:
# Testing electrostatic-only QM/MM optimization

test_water_x = np.array([
    2.5, 0.0, 3.0,
    0.0, 0.0, 0.0
])

test_distance = 3.0

mol_qmmm, E_qm, E_qmmm = optimize_qmmm_at_distance(
    test_mol,
    test_distance,
    test_water_x,
    maxsteps=25
)

print("\nRESULT")
print("Bare QM energy:")
print(E_qm, "Hartree")

print("\nQM/MM energy:")
print(E_qmmm, "Hartree")

E_electrostatic = (
    E_qmmm - E_qm
) * au_to_kJ_conversion

print("\nElectrostatic interaction:")
print(E_electrostatic, "kJ/mol")

coords_final = mol_qmmm.atom_coords(unit="Angstrom")

actual_distance = np.linalg.norm(
    coords_final[0] - coords_final[2]
)

print("\nTarget C-O distance:")
print(test_distance, "Å")

print("Actual C-O distance:")
print(actual_distance, "Å")

print("\nOptimized coordinates:")
print(coords_final)

geometric-optimize called with the following command line:
/home/chemistry/venvs/jupyter/lib/python3.14/site-packages/ipykernel_launcher.py -f /home/chemistry/.local/share/jupyter/runtime/kernel-f483c515-b651-4fa9-ada0-340e3c482655.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%,  **


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -5.000000   0.000000   0.000000    0.000000  0.000000  0.000000
   H  -5.970000   0.000000   0.000000    0.000000  0.000000  0.000000
   C   0.000000   0.000000   0.000000    0.000000  0.000000  0.000000
  Cl   1.780000   0.000000   0.000000    0.000000  0.000000  0.000000
   H  -0.630000   0.630000   0.630000    0.000000  0.000000  0.000000
   H  -0.630000  -0.630000   0.630000    0.000000  0.000000  0.000000
   H  -0.630000   0.000000  -0.890000    0.000000  0.000000  0.000000

WARN: Mole.unit (Angstrom) is changed to Bohr

converged SCF energy = -574.305014520737
--------------- QMMMRHF_Scanner gradients ---------------
         x                y                z
0 O    -0.0099856768    -0.0002517758    -0.0002165533
1 H     0.0075974117     0.0000311981    -0.0000022025
2 C     0.1137643966     0.0001287242    -0.0268861286
3 Cl     0.0061025534    

Step    0 : Gradient = 5.785e-02/9.987e-02 (rms/max) Energy = -574.3050145207
Hessian Eigenvalues: 5.00000e-02 5.00000e-02 5.00000e-02 ... 3.46752e-01 3.47392e-01 5.01282e-01



Geometry optimization cycle 2
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -4.274293   0.000000   0.000000    0.725707  0.000000  0.000000
   H  -5.534537  -0.000000  -0.000000    0.435463 -0.000000 -0.000000
   C  -0.693718   0.000000   0.000296   -0.693718  0.000000  0.000296
  Cl   1.202358   0.000001  -0.046074   -0.577642  0.000001 -0.046074
   H  -1.153006   0.646900   0.647624   -0.523006  0.016900  0.017624
   H  -1.153006  -0.646900   0.647624   -0.523006 -0.016900  0.017624
   H  -1.215553  -0.000000  -0.879472   -0.585553 -0.000000  0.010528
converged SCF energy = -574.302687155996
--------------- QMMMRHF_Scanner gradients ---------------
         x                y                z
0 O     0.0843259683    -0.0002447739    -0.0004576383
1 H    -0.0918013776    -0.0000148858    -0.0001308660
2 C    -0.0076012397     0.0000472231    -0.0234225454
3 Cl     0.0308420441    -0.0008549418     0.0001337971
4 H    -0.0084680

Step    1 : Displace = 5.333e-01/9.746e-01 (rms/max) Trust = 1.000e-01 (=) Grad_T = 6.893e-02/9.180e-02 (rms/max) E (change) = -574.3026871560 (+2.327e-03) Quality = 0.005
Constraint                         Current      Target       Diff.
Distance 1-3                       3.58057     3.00000     0.58057
Hessian Eigenvalues: 4.35783e-03 5.00000e-02 5.00000e-02 ... 3.47324e-01 3.98561e-01 5.72074e-01


...


Geometry optimization cycle 25
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -4.270647   0.628551  -0.046933    0.003024  0.045872  0.123240
   H  -4.641553  -0.210811  -0.367033   -0.014771  0.107038 -0.146765
   C  -1.275621   0.443763  -0.060842    0.007463  0.030980 -0.034931
  Cl  -0.281050   0.884651  -1.650454   -0.081542  0.053216 -0.087869
   H  -0.948403   1.175563   0.660030    0.048571  0.020820 -0.043356
   H  -0.936124  -0.549944   0.182015    0.024270  0.029214 -0.066569
   H  -2.379266   0.522911  -0.256453   -0.005357  0.038131  0.026468
converged SCF energy = -574.407632184264
--------------- QMMMRHF_Scanner gradients ---------------
         x                y                z
0 O    -0.0093466504     0.0018096449     0.0005447686
1 H    -0.0006992718    -0.0009424821    -0.0000311162
2 C     0.0125952112    -0.0013471428    -0.0014199660
3 Cl     0.0002234569    -0.0002247422     0.0007225048
4 H    -0.000344

Step   24 : Displace = 7.790e-02/1.468e-01 (rms/max) Trust = 3.000e-01 (=) Grad_T = 2.856e-03/4.233e-03 (rms/max) E (change) = -574.4076321843 (-2.015e-04) Quality = 1.381
Hessian Eigenvalues: 1.28396e-03 1.63187e-03 8.03249e-03 ... 3.69307e-01 4.82320e-01 6.93932e-01



Geometry optimization cycle 26
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -4.244385   0.597494   0.032997    0.026262 -0.031057  0.079930
   H  -4.697505  -0.085521  -0.486233   -0.055953  0.125291 -0.119200
   C  -1.257571   0.466182  -0.069701    0.018050  0.022420 -0.008860
  Cl  -0.358765   0.935209  -1.704793   -0.077715  0.050559 -0.054339
   H  -0.898942   1.192281   0.638762    0.049461  0.016718 -0.021267
   H  -0.887621  -0.520090   0.150817    0.048503  0.029855 -0.031198
   H  -2.367889   0.527465  -0.208692    0.011377  0.004554  0.047761
converged SCF energy = -574.407946717259
--------------- QMMMRHF_Scanner gradients ---------------
         x                y                z
0 O    -0.0089879728     0.0005292154     0.0000520118
1 H    -0.0011306759    -0.0001958835     0.0002916155
2 C     0.0105445330    -0.0007385185    -0.0025315204
3 Cl    -0.0000570821    -0.0002892401     0.0011768288
4 H    -0.000270

Step   25 : Displace = 8.085e-02/1.499e-01 (rms/max) Trust = 3.000e-01 (=) Grad_T = 1.383e-03/2.256e-03 (rms/max) E (change) = -574.4079467173 (-3.145e-04) Quality = 1.256
Constraint                         Current      Target       Diff.
Distance 1-3                       2.99146     3.00000    -0.00854
Hessian Eigenvalues: 1.28396e-03 1.63187e-03 8.03249e-03 ... 3.69307e-01 4.82320e-01 6.93932e-01
Maximum iterations reached (25); increase --maxiter for more


Geometry optimization failed to converge in 25 iterations
converged SCF energy = -574.409604872825
converged SCF energy = -574.407946725683

RESULT
Bare QM energy:
-574.4096048728248 Hartree

QM/MM energy:
-574.4079467256834 Hartree

Electrostatic interaction:
4.353464689869864 kJ/mol

Target C-O distance:
3.0 Å
Actual C-O distance:
2.991462142141826 Å

Optimized coordinates:
[[-4.24438494  0.59749383  0.0329973 ]
 [-4.69750523 -0.08552063 -0.48623285]
 [-1.25757125  0.46618235 -0.06970149]
 [-0.35876546  0.93520925 -1.70479262]
 [-0.89894233  1.19228122  0.63876249]
 [-0.88762089 -0.52008953  0.15081703]
 [-2.36788875  0.52746514 -0.20869209]]


In [23]:
# Make sure the water position actually matters
coords = mol_qmmm.atom_coords(unit="Angstrom")

test_mol = build_molecule(coords)

water_tests = [
    np.array([2.5, 0.0, 3.0, 0.0, 0.0, 0.0]),
    np.array([2.5, 0.0, 4.0, 0.0, 0.0, 0.0]),
    np.array([3.5, 0.0, 3.0, 0.0, 0.0, 0.0]),
]

for wx in water_tests:

    O, H1, H2, M = water_from_variables(wx)

    mm_coords = np.array([H1, H2, M])
    mm_charges = np.array([0.58, 0.58, -1.16])

    mf_qm = make_scf(test_mol)
    E_qm = mf_qm.kernel()

    mf_qmmm = mm_charge(
        scf.RHF(test_mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )
    mf_qmmm.conv_tol = SCF_CONV_TOL
    E_qmmm = mf_qmmm.kernel()

    E_elec = (
        E_qmmm - E_qm
    ) * au_to_kJ_conversion

    print(
        f"Water O = {O}, "
        f"E_elec = {E_elec:.6f} kJ/mol"
    )

converged SCF energy = -574.409604872824
converged SCF energy = -574.407946725681
Water O = [2.5 0.  3. ], E_elec = 4.353465 kJ/mol
converged SCF energy = -574.409604872825
converged SCF energy = -574.408186882975
Water O = [2.5 0.  4. ], E_elec = 3.722932 kJ/mol
converged SCF energy = -574.409604872824
converged SCF energy = -574.408088122773
Water O = [3.5 0.  3. ], E_elec = 3.982227 kJ/mol


In [24]:
# Verifying that the LJ energy behaves sensibly for these same three water positions.
for wx in water_tests:

    O, H1, H2, M = water_from_variables(wx)

    E_LJ = qm_water_lj_energy(
        test_mol.atom_coords(unit="Angstrom"),
        np.array([O]),
        qm_atom_types
    )

    print(
        f"Water O = {O}, "
        f"E_LJ = {E_LJ:.6f} kJ/mol"
    )

Water O = [2.5 0.  3. ], E_LJ = -0.532886 kJ/mol
Water O = [2.5 0.  4. ], E_LJ = -0.253098 kJ/mol
Water O = [3.5 0.  3. ], E_LJ = -0.280669 kJ/mol


In [25]:
for wx in water_tests:

    O, H1, H2, M = water_from_variables(wx)

    mm_coords = np.array([H1, H2, M])
    mm_charges = np.array([0.58, 0.58, -1.16])

    # Bare QM
    mf_qm = make_scf(test_mol)
    E_qm = mf_qm.kernel()

    # QM/MM electrostatics
    mf_qmmm = mm_charge(
        scf.RHF(test_mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )
    mf_qmmm.conv_tol = SCF_CONV_TOL
    E_qmmm = mf_qmmm.kernel()

    E_elec = (
        E_qmmm - E_qm
    ) * au_to_kJ_conversion

    # LJ
    E_LJ = qm_water_lj_energy(
        test_mol.atom_coords(unit="Angstrom"),
        np.array([O]),
        qm_atom_types
    )

    print("\nWater O:", O)
    print(f"Electrostatic: {E_elec:.6f} kJ/mol")
    print(f"LJ:            {E_LJ:.6f} kJ/mol")
    print(f"Total:         {E_elec + E_LJ:.6f} kJ/mol")

converged SCF energy = -574.409604872824
converged SCF energy = -574.407946725681

Water O: [2.5 0.  3. ]
Electrostatic: 4.353465 kJ/mol
LJ:            -0.532886 kJ/mol
Total:         3.820579 kJ/mol
converged SCF energy = -574.409604872824
converged SCF energy = -574.408186882974

Water O: [2.5 0.  4. ]
Electrostatic: 3.722932 kJ/mol
LJ:            -0.253098 kJ/mol
Total:         3.469834 kJ/mol
converged SCF energy = -574.409604872825
converged SCF energy = -574.408088122773

Water O: [3.5 0.  3. ]
Electrostatic: 3.982227 kJ/mol
LJ:            -0.280669 kJ/mol
Total:         3.701558 kJ/mol


In [27]:
# Testing the gradient function
qm_coords_A = test_mol.atom_coords(unit="Angstrom")

water_O = np.array([2.5, 0.0, 3.0])

E_old = qm_water_lj_energy(
    qm_coords_A,
    np.array([water_O]),
    qm_atom_types
)

E_new, grad = qm_water_lj_energy_gradient(
    qm_coords_A,
    np.array([water_O]),
    qm_atom_types
)

print("Old LJ energy:", E_old)
print("New LJ energy:", E_new)
print("Difference:", E_new - E_old)

print("\nLJ gradient (kJ/mol/Angstrom):")
print(grad)

Old LJ energy: -0.5328857302341696
New LJ energy: -0.5328857302341696
Difference: 0.0

LJ gradient (kJ/mol/Angstrom):
[[-6.90163189e-03  6.11424546e-04 -3.03617908e-03]
 [-1.71312639e-05 -2.03553382e-07 -8.29781611e-06]
 [-1.53940835e-01  1.90986401e-02 -1.25760067e-01]
 [-1.41818938e-01  4.63942862e-02 -2.33397493e-01]
 [-1.39596399e-02  4.89676344e-03 -9.69773009e-03]
 [-1.08188638e-02 -1.66098214e-03 -9.09928335e-03]
 [-1.89171498e-03  2.04978743e-04 -1.24693295e-03]]


In [28]:
# checks that the gradient is actually the derivative of the LJ energy.
delta = 1e-5

i = 0       # QM atom
j = 0       # x coordinate

coords_plus = qm_coords_A.copy()
coords_minus = qm_coords_A.copy()

coords_plus[i, j] += delta
coords_minus[i, j] -= delta

E_plus = qm_water_lj_energy(
    coords_plus,
    np.array([water_O]),
    qm_atom_types
)

E_minus = qm_water_lj_energy(
    coords_minus,
    np.array([water_O]),
    qm_atom_types
)

numerical_gradient = (
    E_plus - E_minus
) / (2 * delta)

print("Analytic gradient:",
      grad[i, j])

print("Numerical gradient:",
      numerical_gradient)

print("Difference:",
      grad[i, j] - numerical_gradient)

Analytic gradient: -0.006901631885410815
Numerical gradient: -0.006901631882660907
Difference: -2.7499079402470983e-12


In [32]:
# Testing whether QM/MM optimization works

# ------------------------------------------------
# Test optimize_qmmm_at_distance()
# ------------------------------------------------

test_mol = build_molecule(
    np.array([
        [-5.000,  0.000,  0.000],
        [-5.970,  0.000,  0.000],
        [ 0.000,  0.000,  0.000],
        [ 1.780,  0.000,  0.000],
        [-0.630,  0.630,  0.630],
        [-0.630, -0.630,  0.630],
        [-0.630,  0.000, -0.890],
    ])
)

target_distance = 3.0

# Save initial coordinates
initial_coords = test_mol.atom_coords(
    unit="Angstrom"
).copy()

# Initial bare QM energy
mf_qm_initial = make_scf(test_mol)
E_qm_initial = mf_qm_initial.kernel()

print("\nInitial bare QM energy:")
print(E_qm_initial, "Hartree")

# ------------------------------------------------
# Call the actual function
# ------------------------------------------------

optimized_mol, E_qmmm = optimize_qmmm_at_distance(
    test_mol,
    target_distance,
    water_x,
    maxsteps=25
)

# ------------------------------------------------
# Final bare QM energy
# ------------------------------------------------

mf_qm_final = make_scf(optimized_mol)
E_qm_final = mf_qm_final.kernel()

# ------------------------------------------------
# Final coordinates
# ------------------------------------------------

final_coords = optimized_mol.atom_coords(
    unit="Angstrom"
)

# Actual C-O distance
actual_distance = np.linalg.norm(
    final_coords[0] - final_coords[2]
)

# ------------------------------------------------
# Report
# ------------------------------------------------

print("\n" + "=" * 60)
print("RESULT")
print("=" * 60)

print(f"Target C-O distance: {target_distance:.6f} Å")
print(f"Actual C-O distance: {actual_distance:.8f} Å")

print("\nInitial bare QM energy:")
print(f"{E_qm_initial:.12f} Hartree")

print("\nFinal bare QM energy:")
print(f"{E_qm_final:.12f} Hartree")

print("\nFinal QM/MM energy:")
print(f"{E_qmmm:.12f} Hartree")

print("\nQM/MM - bare QM:")
print(
    f"{(E_qmmm - E_qm_final) * au_to_kJ_conversion:.6f} kJ/mol"
)

print("\nInitial coordinates:")
print(initial_coords)

print("\nOptimized coordinates:")
print(final_coords)

converged SCF energy = -574.308632234646


geometric-optimize called with the following command line:
/home/chemistry/venvs/jupyter/lib/python3.14/site-packages/ipykernel_launcher.py -f /home/chemistry/.local/share/jupyter/runtime/kernel-f483c515-b651-4fa9-ada0-340e3c482655.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%,  **


Initial bare QM energy:
-574.3086322346464 Hartree

Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -5.000000   0.000000   0.000000    0.000000  0.000000  0.000000
   H  -5.970000   0.000000   0.000000    0.000000  0.000000  0.000000
   C   0.000000   0.000000   0.000000    0.000000  0.000000  0.000000
  Cl   1.780000   0.000000   0.000000    0.000000  0.000000  0.000000
   H  -0.630000   0.630000   0.630000    0.000000  0.000000  0.000000
   H  -0.630000  -0.630000   0.630000    0.000000  0.000000  0.000000
   H  -0.630000   0.000000  -0.890000    0.000000  0.000000  0.000000

WARN: Mole.unit (Angstrom) is changed to Bohr

converged SCF energy = -574.305014520736
--------------- QMMMRHF_Scanner gradients ---------------
         x                y                z
0 O    -0.0099856768    -0.0002517758    -0.0002165533
1 H     0.0075974117     0.0000311981    -0.0000022025
2 C     0.1137643966     0.0

Step    0 : Gradient = 5.785e-02/9.987e-02 (rms/max) Energy = -574.3050145207
Hessian Eigenvalues: 5.00000e-02 5.00000e-02 5.00000e-02 ... 3.46752e-01 3.47392e-01 5.01282e-01



Geometry optimization cycle 2
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -4.274293   0.000000   0.000000    0.725707  0.000000  0.000000
   H  -5.534537  -0.000000  -0.000000    0.435463 -0.000000 -0.000000
   C  -0.693718   0.000000   0.000296   -0.693718  0.000000  0.000296
  Cl   1.202358   0.000001  -0.046074   -0.577642  0.000001 -0.046074
   H  -1.153006   0.646900   0.647624   -0.523006  0.016900  0.017624
   H  -1.153006  -0.646900   0.647624   -0.523006 -0.016900  0.017624
   H  -1.215553  -0.000000  -0.879472   -0.585553 -0.000000  0.010528
converged SCF energy = -574.302687155996
--------------- QMMMRHF_Scanner gradients ---------------
         x                y                z
0 O     0.0843259683    -0.0002447739    -0.0004576383
1 H    -0.0918013776    -0.0000148858    -0.0001308660
2 C    -0.0076012397     0.0000472231    -0.0234225454
3 Cl     0.0308420441    -0.0008549418     0.0001337971
4 H    -0.0084680

...


Geometry optimization cycle 25
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -4.270647   0.628551  -0.046933    0.003024  0.045872  0.123240
   H  -4.641553  -0.210811  -0.367033   -0.014771  0.107038 -0.146765
   C  -1.275621   0.443763  -0.060842    0.007463  0.030980 -0.034931
  Cl  -0.281050   0.884650  -1.650454   -0.081542  0.053216 -0.087869
   H  -0.948403   1.175563   0.660030    0.048571  0.020820 -0.043356
   H  -0.936124  -0.549944   0.182015    0.024270  0.029214 -0.066569
   H  -2.379266   0.522911  -0.256453   -0.005357  0.038130  0.026468
converged SCF energy = -574.407632184225
--------------- QMMMRHF_Scanner gradients ---------------
         x                y                z
0 O    -0.0093466529     0.0018096383     0.0005447644
1 H    -0.0006992697    -0.0009424760    -0.0000311133
2 C     0.0125952164    -0.0013471434    -0.0014199683
3 Cl     0.0002234570    -0.0002247424     0.0007225039
4 H    -0.000344

Step   24 : Displace = 7.790e-02/1.468e-01 (rms/max) Trust = 3.000e-01 (=) Grad_T = 2.856e-03/4.233e-03 (rms/max) E (change) = -574.4076321842 (-2.015e-04) Quality = 1.381
Hessian Eigenvalues: 1.28396e-03 1.63187e-03 8.03249e-03 ... 3.69307e-01 4.82319e-01 6.93932e-01



Geometry optimization cycle 26
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -4.244385   0.597494   0.032997    0.026262 -0.031057  0.079930
   H  -4.697505  -0.085521  -0.486233   -0.055952  0.125291 -0.119200
   C  -1.257571   0.466182  -0.069702    0.018049  0.022420 -0.008860
  Cl  -0.358765   0.935209  -1.704793   -0.077715  0.050559 -0.054339
   H  -0.898942   1.192281   0.638762    0.049461  0.016718 -0.021267
   H  -0.887621  -0.520090   0.150817    0.048503  0.029855 -0.031198
   H  -2.367889   0.527465  -0.208692    0.011377  0.004554  0.047761
converged SCF energy = -574.407946717092
--------------- QMMMRHF_Scanner gradients ---------------
         x                y                z
0 O    -0.0089879720     0.0005292198     0.0000520154
1 H    -0.0011306766    -0.0001958870     0.0002916126
2 C     0.0105445402    -0.0007385209    -0.0025315193
3 Cl    -0.0000570819    -0.0002892404     0.0011768280
4 H    -0.000270

Step   25 : Displace = 8.085e-02/1.499e-01 (rms/max) Trust = 3.000e-01 (=) Grad_T = 1.383e-03/2.256e-03 (rms/max) E (change) = -574.4079467171 (-3.145e-04) Quality = 1.256
Constraint                         Current      Target       Diff.
Distance 1-3                       2.99146     3.00000    -0.00854
Hessian Eigenvalues: 1.28396e-03 1.63187e-03 8.03249e-03 ... 3.69307e-01 4.82319e-01 6.93932e-01
Maximum iterations reached (25); increase --maxiter for more


Geometry optimization failed to converge in 25 iterations
converged SCF energy = -574.407946725514
converged SCF energy = -574.40960487261

RESULT
Target C-O distance: 3.000000 Å
Actual C-O distance: 2.99146217 Å

Initial bare QM energy:
-574.308632234646 Hartree

Final bare QM energy:
-574.409604872610 Hartree

Final QM/MM energy:
-574.407946725514 Hartree

QM/MM - bare QM:
4.353465 kJ/mol

Initial coordinates:
[[-5.    0.    0.  ]
 [-5.97  0.    0.  ]
 [ 0.    0.    0.  ]
 [ 1.78  0.    0.  ]
 [-0.63  0.63  0.63]
 [-0.63 -0.63  0.63]
 [-0.63  0.   -0.89]]

Optimized coordinates:
[[-4.24438498  0.5974939   0.0329973 ]
 [-4.69750513 -0.08552069 -0.48623282]
 [-1.25757127  0.46618237 -0.06970151]
 [-0.35876548  0.93520924 -1.70479265]
 [-0.89894234  1.19228125  0.63876247]
 [-0.88762094 -0.52008952  0.15081701]
 [-2.36788879  0.52746518 -0.20869211]]


In [34]:
# Testing the gradient
# Build the current TIP4P-D water
O, H1, H2, M = water_from_variables(water_x)

mm_coords = np.array([
    H1,
    H2,
    M
])

mm_charges = np.array([
    0.58,
    0.58,
    -1.16
])

# QM/MM SCF
mf_qmmm = mm_charge(
    scf.RHF(test_mol),
    mm_coords,
    mm_charges,
    unit="Angstrom"
)

mf_qmmm.conv_tol = SCF_CONV_TOL

mf_qmmm.kernel()

# Build combined gradient
grad = QMMM_LJ_Gradients(
    mf_qmmm,
    O,
    qm_atom_types
)

# Calculate combined gradient
combined_gradient = grad.kernel()

print("Combined gradient:")
print(combined_gradient)

print("\nGradient shape:")
print(combined_gradient.shape)

print("\nMaximum absolute gradient:")
print(np.max(np.abs(combined_gradient)))

converged SCF energy = -574.305014520737
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0099856768    -0.0002517758    -0.0002165533
1 H     0.0075974117     0.0000311981    -0.0000022025
2 C     0.1137643966     0.0001287242    -0.0268861286
3 Cl     0.0061025534    -0.0007342618    -0.0000192679
4 H    -0.0406633867    -0.0645841297     0.0069044847
5 H    -0.0413168782     0.0647130735     0.0073871398
6 H    -0.0356751281     0.0000087939     0.0142266191
----------------------------------------------
Combined gradient:
[[-9.98630024e-03 -2.51983634e-04 -2.16761107e-04]
 [ 7.59741036e-03  3.11977116e-05 -2.20294474e-06]
 [ 1.13727641e-01  9.19686078e-05 -2.69228843e-02]
 [ 7.09173396e-03  2.70039283e-03  3.41538679e-03]
 [-4.06670828e-02 -6.45863379e-02  6.90227649e-03]
 [-4.13180205e-02  6.47119312e-02  7.38645734e-03]
 [-3.56757065e-02  8.33189214e-06  1.42259927e-02]]

Gradient shape:
(7, 3)

Maximum absolute gradient:
0.

In [40]:
# Re-running the gradient test
O, H1, H2, M = water_from_variables(water_x)

mm_coords = np.array([
    H1,
    H2,
    M
])

mm_charges = np.array([
    0.58,
    0.58,
    -1.16
])

mf_qmmm = mm_charge(
    scf.RHF(test_mol),
    mm_coords,
    mm_charges,
    unit="Angstrom"
)

mf_qmmm.conv_tol = SCF_CONV_TOL
mf_qmmm.kernel()

grad = QMMM_LJ_Gradients(
    mf_qmmm,
    O,
    qm_atom_types
)

combined_gradient = grad.kernel()

print("Combined gradient:")
print(combined_gradient)

print("\nShape:")
print(combined_gradient.shape)

print("\nMax gradient:")
print(np.max(np.abs(combined_gradient)))

converged SCF energy = -574.305014520737
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0099856768    -0.0002517758    -0.0002165533
1 H     0.0075974117     0.0000311981    -0.0000022025
2 C     0.1137643966     0.0001287242    -0.0268861286
3 Cl     0.0061025534    -0.0007342618    -0.0000192679
4 H    -0.0406633867    -0.0645841297     0.0069044847
5 H    -0.0413168782     0.0647130735     0.0073871398
6 H    -0.0356751281     0.0000087939     0.0142266191
----------------------------------------------
Combined gradient:
[[-9.98630024e-03 -2.51983634e-04 -2.16761107e-04]
 [ 7.59741036e-03  3.11977116e-05 -2.20294474e-06]
 [ 1.13727641e-01  9.19686078e-05 -2.69228843e-02]
 [ 7.09173396e-03  2.70039283e-03  3.41538679e-03]
 [-4.06670828e-02 -6.45863379e-02  6.90227649e-03]
 [-4.13180205e-02  6.47119312e-02  7.38645734e-03]
 [-3.56757065e-02  8.33189214e-06  1.42259927e-02]]

Shape:
(7, 3)

Max gradient:
0.11372764092105353


In [43]:
# # Testing optimize_qmmm_lj_at_distance at one frame (this failed)
# test_mol = build_molecule(
#     np.array([
#         [-5.000,  0.000,  0.000],
#         [-5.970,  0.000,  0.000],
#         [ 0.000,  0.000,  0.000],
#         [ 1.780,  0.000,  0.000],
#         [-0.630,  0.630,  0.630],
#         [-0.630, -0.630,  0.630],
#         [-0.630,  0.000, -0.890],
#     ])
# )

# optimized_mol, energy_qm, energy_qmmm, E_LJ = \
#     optimize_qmmm_lj_at_distance(
#         test_mol,
#         3.0,
#         water_x,
#         maxsteps=25
#     )

# coords = optimized_mol.atom_coords(
#     unit="Angstrom"
# )

# actual_distance = np.linalg.norm(
#     coords[0] - coords[2]
# )

# print("\n" + "=" * 60)
# print("QM/MM + LJ OPTIMIZATION")
# print("=" * 60)

# print(f"Target C-O: {3.0:.6f} Å")
# print(f"Actual C-O: {actual_distance:.8f} Å")

# print(f"\nBare QM:    {energy_qm:.12f} Hartree")
# print(f"QM/MM:      {energy_qmmm:.12f} Hartree")
# print(f"LJ:         {E_LJ:.6f} kJ/mol")

# print(
#     "\nElectrostatic interaction:",
#     (energy_qmmm - energy_qm)
#     * au_to_kJ_conversion,
#     "kJ/mol"
# )

In [44]:
test_mol = build_molecule(
    np.array([
        [-5.000,  0.000,  0.000],
        [-5.970,  0.000,  0.000],
        [ 0.000,  0.000,  0.000],
        [ 1.780,  0.000,  0.000],
        [-0.630,  0.630,  0.630],
        [-0.630, -0.630,  0.630],
        [-0.630,  0.000, -0.890],
    ])
)

E_qmmm, grad_qmmm = qmmm_gradient(
    test_mol,
    water_x
)

print("QM/MM energy:")
print(E_qmmm, "Hartree")

print("\nGradient:")
print(grad_qmmm)

print("\nMaximum gradient:")
print(np.max(np.abs(grad_qmmm)))

converged SCF energy = -574.305014520736
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0099856768    -0.0002517758    -0.0002165533
1 H     0.0075974117     0.0000311981    -0.0000022025
2 C     0.1137643966     0.0001287242    -0.0268861286
3 Cl     0.0061025534    -0.0007342618    -0.0000192679
4 H    -0.0406633867    -0.0645841297     0.0069044847
5 H    -0.0413168782     0.0647130735     0.0073871398
6 H    -0.0356751281     0.0000087939     0.0142266191
----------------------------------------------
QM/MM energy:
-574.3050145207363 Hartree

Gradient:
[[-9.98567682e-03 -2.51775827e-04 -2.16553300e-04]
 [ 7.59741171e-03  3.11981103e-05 -2.20254605e-06]
 [ 1.13764397e-01  1.28724244e-04 -2.68861286e-02]
 [ 6.10255342e-03 -7.34261834e-04 -1.92678762e-05]
 [-4.06633867e-02 -6.45841297e-02  6.90448473e-03]
 [-4.13168782e-02  6.47130735e-02  7.38713980e-03]
 [-3.56751281e-02  8.79386983e-06  1.42266191e-02]]

Maximum gradient:
0.

In [46]:
# coords0 = test_mol.atom_coords(unit="Angstrom")

# result = minimize(
#     lambda x: qmmm_objective(x, water_x),
#     coords0.reshape(-1),
#     jac=True,
#     method="BFGS"
# )

# print(result)

converged SCF energy = -574.305014520737
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0099856768    -0.0002517758    -0.0002165533
1 H     0.0075974117     0.0000311981    -0.0000022025
2 C     0.1137643966     0.0001287242    -0.0268861286
3 Cl     0.0061025534    -0.0007342618    -0.0000192679
4 H    -0.0406633867    -0.0645841297     0.0069044847
5 H    -0.0413168782     0.0647130735     0.0073871398
6 H    -0.0356751281     0.0000087939     0.0142266191
----------------------------------------------
converged SCF energy = -574.34114830215
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O     0.0046849347    -0.0002457484    -0.0001155427
1 H    -0.0070441848     0.0000261303    -0.0000073109
2 C     0.0077322801    -0.0001208711    -0.0048227681
3 Cl     0.0297836733    -0.0009709099    -0.0002259155
4 H    -0.0125619973    -0.0706400472    -0.0105004890
5 H    -0.0129721359

KeyboardInterrupt: 

In [51]:
optimized_mol, energy_qm, energy_qmmm, result = \
    optimize_qmmm_at_distance(
        test_mol,
        3.0,
        water_x,
        maxiter=25
    )

coords_opt = optimized_mol.atom_coords(
    unit="Angstrom"
)

actual_distance = np.linalg.norm(
    coords_opt[0] - coords_opt[2]
)

print("\nRESULT")
print("Optimization success:", result.success)
print("Message:", result.message)

print("\nTarget C-O distance:")
print(3.0, "Å")

print("\nActual C-O distance:")
print(actual_distance, "Å")

print("\nBare QM energy:")
print(energy_qm, "Hartree")

print("\nQM/MM energy:")
print(energy_qmmm, "Hartree")

print("\nQM/MM - bare QM:")
print(
    (energy_qmmm - energy_qm) * au_to_kJ_conversion,
    "kJ/mol"
)

converged SCF energy = -574.305014520737
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0099856768    -0.0002517758    -0.0002165533
1 H     0.0075974117     0.0000311981    -0.0000022025
2 C     0.1137643966     0.0001287242    -0.0268861286
3 Cl     0.0061025534    -0.0007342618    -0.0000192679
4 H    -0.0406633867    -0.0645841297     0.0069044847
5 H    -0.0413168782     0.0647130735     0.0073871398
6 H    -0.0356751281     0.0000087939     0.0142266191
----------------------------------------------
converged SCF energy = -574.082851396975
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O     0.0418749018    -0.0001482973    -0.0002002182
1 H    -0.0618089289    -0.0000563558    -0.0001583509
2 C    -0.0498183995     0.0000772278    -0.0118823421
3 Cl     0.0157814625    -0.0024563492     0.0028082103
4 H     0.0174428821    -0.1140048632    -0.0373474703
5 H     0.017082205

In [54]:
# test qmmm_lj_objective
E_test, grad_test = qmmm_lj_objective(
    test_mol.atom_coords(unit="Angstrom").reshape(-1),
    water_x
)

print("QM/MM + LJ energy:")
print(E_test, "Hartree")

print("\nGradient shape:")
print(grad_test.shape)

print("\nMaximum gradient:")
print(np.max(np.abs(grad_test)))


coords_A = test_mol.atom_coords(unit="Angstrom")

E_qmmm, grad_qmmm = qmmm_gradient(
    test_mol,
    water_x
)

O, H1, H2, M = water_from_variables(water_x)

E_LJ, grad_LJ = qm_water_lj_energy_gradient(
    coords_A,
    np.array([O]),
    qm_atom_types
)

print("QM/MM:")
print(E_qmmm, "Hartree")

print("LJ:")
print(E_LJ, "kJ/mol")

print("LJ:")
print(E_LJ / au_to_kJ_conversion, "Hartree")

print("Total:")
print(E_qmmm + E_LJ / au_to_kJ_conversion, "Hartree")

converged SCF energy = -574.305014520737
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0099856768    -0.0002517758    -0.0002165533
1 H     0.0075974117     0.0000311981    -0.0000022025
2 C     0.1137643966     0.0001287242    -0.0268861286
3 Cl     0.0061025534    -0.0007342618    -0.0000192679
4 H    -0.0406633867    -0.0645841297     0.0069044847
5 H    -0.0413168782     0.0647130735     0.0073871398
6 H    -0.0356751281     0.0000087939     0.0142266191
----------------------------------------------
QM/MM + LJ energy:
-574.2854141029162 Hartree

Gradient shape:
(21,)

Maximum gradient:
0.0840267892838329
converged SCF energy = -574.305014520736
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0099856768    -0.0002517758    -0.0002165533
1 H     0.0075974117     0.0000311981    -0.0000022025
2 C     0.1137643966     0.0001287242    -0.0268861286
3 Cl     0.0061025534  

In [65]:
print("Testing QMMM_LJ_Gradients interface...")

O, H1, H2, M = water_from_variables(water_x)

mm_coords = np.array([
    H1,
    H2,
    M
])

mm_charges = np.array([
    0.58,
    0.58,
    -1.16
])

mf = mm_charge(
    scf.RHF(test_mol),
    mm_coords,
    mm_charges,
    unit="Angstrom"
)

mf.conv_tol = SCF_CONV_TOL

grad = QMMM_LJ_Gradients(
    mf,
    O,
    qm_atom_types,
    mm_coords,
    mm_charges
)

print("nuc_grad_method:", grad.nuc_grad_method())
print("as_scanner:", grad.as_scanner())
print("converged:", grad.converged)

E, G = grad(test_mol)

print("\nEnergy:", E)
print("Gradient shape:", G.shape)
print("Maximum gradient:", np.max(np.abs(G)))

Testing QMMM_LJ_Gradients interface...
nuc_grad_method: <__main__.QMMM_LJ_Gradients object at 0x7fcc71917c50>
as_scanner: <__main__.QMMM_LJ_Gradients object at 0x7fcc71917c50>
converged: True
converged SCF energy = -574.305014520737
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0099856768    -0.0002517758    -0.0002165533
1 H     0.0075974117     0.0000311981    -0.0000022025
2 C     0.1137643966     0.0001287242    -0.0268861286
3 Cl     0.0061025534    -0.0007342618    -0.0000192679
4 H    -0.0406633867    -0.0645841297     0.0069044847
5 H    -0.0413168782     0.0647130735     0.0073871398
6 H    -0.0356751281     0.0000087939     0.0142266191
----------------------------------------------

Energy: -574.3032847081514
Gradient shape: (7, 3)
Maximum gradient: 0.11372764092104909


In [67]:
# Optimizing the QM geometry with fixed water and lj
optimized_mol, energy_qm, energy_qmmm, E_LJ, result = \
    optimize_qmmm_lj_at_distance(
        test_mol,
        3.0,
        water_x,
        maxsteps=100
    )

geometric-optimize called with the following command line:
/home/chemistry/venvs/jupyter/lib/python3.14/site-packages/ipykernel_launcher.py -f /home/chemistry/.local/share/jupyter/runtime/kernel-f483c515-b651-4fa9-ada0-340e3c482655.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%,  **


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -5.000000   0.000000   0.000000    0.000000  0.000000  0.000000
   H  -5.970000   0.000000   0.000000    0.000000  0.000000  0.000000
   C   0.000000   0.000000   0.000000    0.000000  0.000000  0.000000
  Cl   1.780000   0.000000   0.000000    0.000000  0.000000  0.000000
   H  -0.630000   0.630000   0.630000    0.000000  0.000000  0.000000
   H  -0.630000  -0.630000   0.630000    0.000000  0.000000  0.000000
   H  -0.630000   0.000000  -0.890000    0.000000  0.000000  0.000000

WARN: Mole.unit (Angstrom) is changed to Bohr

converged SCF energy = -574.305014520736
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0099856768    -0.0002517758    -0.0002165533
1 H     0.0075974117     0.0000311981    -0.0000022025
2 C     0.1137643966     0.0001287242    -0.0268861286
3 Cl     0.0061025534    -0.00073

Step    0 : Gradient = 5.777e-02/9.937e-02 (rms/max) Energy = -574.3032847082
Hessian Eigenvalues: 5.00000e-02 5.00000e-02 5.00000e-02 ... 3.46752e-01 3.47392e-01 5.01282e-01



Geometry optimization cycle 2
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -4.274295   0.000000   0.000000    0.725705  0.000000  0.000000
   H  -5.534532  -0.000000  -0.000000    0.435468 -0.000000 -0.000000
   C  -0.693720  -0.000000   0.000296   -0.693720 -0.000000  0.000296
  Cl   1.202355  -0.000002  -0.046077   -0.577645 -0.000002 -0.046077
   H  -1.153006   0.646904   0.647624   -0.523006  0.016904  0.017624
   H  -1.153006  -0.646904   0.647624   -0.523006 -0.016904  0.017624
   H  -1.215554  -0.000000  -0.879472   -0.585554 -0.000000  0.010528
converged SCF energy = -574.302689673903
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O     0.0843318903    -0.0002452037    -0.0004579986
1 H    -0.0918079871    -0.0000144618    -0.0001305186
2 C    -0.0075880211     0.0000471098    -0.0234283306
3 Cl     0.0308354807    -0.0008551885     0.0001361600
4 H    -0.0084693256    -

Step    1 : Displace = 5.333e-01/9.746e-01 (rms/max) Trust = 1.000e-01 (=) Grad_T = 6.903e-02/9.181e-02 (rms/max) E (change) = -574.3021349240 (+1.150e-03) Quality = 0.002
Constraint                         Current      Target       Diff.
Distance 1-3                       3.58057     3.00000     0.58057
Hessian Eigenvalues: 4.36660e-03 5.00000e-02 5.00000e-02 ... 3.47316e-01 3.98512e-01 5.71075e-01


...


Geometry optimization cycle 100
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -3.077956   6.227782  -0.518976    0.029684 -0.020119  0.019048
   H  -3.494989   5.932610   0.303845   -0.001314  0.063672  0.035007
   C  -0.097061   6.049864  -0.771793    0.022359  0.051760 -0.005322
  Cl   1.039085   6.164096   0.787182    0.023371  0.073271 -0.011541
   H   0.148431   5.091169  -1.192231    0.048447  0.061868 -0.016101
   H   0.245195   6.859157  -1.391218   -0.007270  0.068108 -0.002993
   H  -1.172551   6.139475  -0.481040    0.021187  0.018867  0.002066
converged SCF energy = -574.411584811024
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0100327293     0.0000745445     0.0016340223
1 H    -0.0009654886     0.0002718901    -0.0008528114
2 C     0.0089662754    -0.0001611580     0.0000917834
3 Cl     0.0011408126     0.0002277576    -0.0001582372
4 H     0.0001666477   

Step   99 : Displace = 3.037e-02/5.665e-02 (rms/max) Trust = 3.000e-01 (=) Grad_T = 6.387e-04/1.070e-03 (rms/max) E (change) = -574.4118329363 (-1.565e-04) Quality = 1.003
Constraint                         Current      Target       Diff.
Distance 1-3                       2.99688     3.00000    -0.00312
Hessian Eigenvalues: 4.40372e-05 2.11509e-03 1.00155e-02 ... 3.70592e-01 3.89341e-01 7.08431e-01



Geometry optimization cycle 101
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -3.018417   6.302517  -0.479545    0.059539  0.074735  0.039432
   H  -3.445327   5.987790   0.331079    0.049662  0.055180  0.027235
   C  -0.038792   6.093655  -0.738322    0.058269  0.043791  0.033471
  Cl   1.099093   6.195460   0.818925    0.060009  0.031364  0.031743
   H   0.202556   5.136380  -1.164719    0.054124  0.045211  0.027512
   H   0.304557   6.905149  -1.354446    0.059362  0.045992  0.036772
   H  -1.112453   6.188922  -0.444591    0.060098  0.049447  0.036449
converged SCF energy = -574.411578746237
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0099233447     0.0002292378     0.0015595894
1 H    -0.0010754111     0.0002609361    -0.0007341702
2 C     0.0088947674    -0.0004460172     0.0002298500
3 Cl     0.0009520681     0.0002355566    -0.0002800031
4 H     0.0003413530   

Step  100 : Displace = 6.795e-03/1.181e-02 (rms/max) Trust = 3.000e-01 (=) Grad_T = 5.677e-04/8.604e-04 (rms/max) E (change) = -574.4118256939 (+7.242e-06) Quality = 0.288
Constraint                         Current      Target       Diff.
Distance 1-3                       2.99813     3.00000    -0.00187
Hessian Eigenvalues: 4.40372e-05 2.11509e-03 1.00155e-02 ... 3.70592e-01 3.89341e-01 7.08431e-01
Maximum iterations reached (100); increase --maxiter for more


Geometry optimization failed to converge in 100 iterations
converged SCF energy = -574.409682061151
converged SCF energy = -574.411578746237


In [68]:
# ------------------------------------------------------------
# Verify TOTAL QM/MM + LJ gradient numerically
# ------------------------------------------------------------

O, H1, H2, M = water_from_variables(water_x)

mm_coords = np.array([
    H1,
    H2,
    M
])

mm_charges = np.array([
    0.58,
    0.58,
    -1.16
])

# Build the gradient object
mf = mm_charge(
    scf.RHF(test_mol),
    mm_coords,
    mm_charges,
    unit="Angstrom"
)

mf.conv_tol = SCF_CONV_TOL

grad_object = QMMM_LJ_Gradients(
    mf,
    O,
    qm_atom_types,
    mm_coords,
    mm_charges
)

# Analytic energy and gradient
E0, G0 = grad_object(test_mol)

print("Analytic total energy:")
print(E0)

print("\nAnalytic gradient:")
print(G0)

# ------------------------------------------------------------
# Numerical derivative for one coordinate
# ------------------------------------------------------------

atom = 0
coord = 0
delta_A = 1.0e-4

coords_plus = test_mol.atom_coords(unit="Angstrom")
coords_minus = test_mol.atom_coords(unit="Angstrom")

coords_plus[atom, coord] += delta_A
coords_minus[atom, coord] -= delta_A

mol_plus = build_molecule(coords_plus)
mol_minus = build_molecule(coords_minus)

E_plus, _ = grad_object(mol_plus)
E_minus, _ = grad_object(mol_minus)

# Energy is Hartree, displacement is Angstrom
numerical = (
    E_plus - E_minus
) / (2.0 * delta_A)

# Convert Hartree/Angstrom -> Hartree/Bohr
numerical_Ha_Bohr = numerical * 1.889726125

print("\nGradient check for atom", atom, "coordinate", coord)
print("Analytic: ", G0[atom, coord])
print("Numerical: ", numerical_Ha_Bohr)
print("Difference:",
      G0[atom, coord] - numerical_Ha_Bohr)

converged SCF energy = -574.305014520737
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0099856768    -0.0002517758    -0.0002165533
1 H     0.0075974117     0.0000311981    -0.0000022025
2 C     0.1137643966     0.0001287242    -0.0268861286
3 Cl     0.0061025534    -0.0007342618    -0.0000192679
4 H    -0.0406633867    -0.0645841297     0.0069044847
5 H    -0.0413168782     0.0647130735     0.0073871398
6 H    -0.0356751281     0.0000087939     0.0142266191
----------------------------------------------
Analytic total energy:
-574.3032847081512

Analytic gradient:
[[-9.98630024e-03 -2.51983634e-04 -2.16761107e-04]
 [ 7.59741036e-03  3.11977116e-05 -2.20294474e-06]
 [ 1.13727641e-01  9.19686078e-05 -2.69228843e-02]
 [ 7.09173396e-03  2.70039283e-03  3.41538679e-03]
 [-4.06670828e-02 -6.45863379e-02  6.90227649e-03]
 [-4.13180205e-02  6.47119312e-02  7.38645734e-03]
 [-3.56757065e-02  8.33189214e-06  1.42259927e-02]]
converged S

In [69]:
# ------------------------------------------------------------
# Separate QM/MM and LJ gradient checks
# ------------------------------------------------------------

O, H1, H2, M = water_from_variables(water_x)

mm_coords = np.array([
    H1,
    H2,
    M
])

mm_charges = np.array([
    0.58,
    0.58,
    -1.16
])

# QM/MM object
mf_qmmm = mm_charge(
    scf.RHF(test_mol),
    mm_coords,
    mm_charges,
    unit="Angstrom"
)

mf_qmmm.conv_tol = SCF_CONV_TOL

E_qmmm = mf_qmmm.kernel()

grad_qmmm = mf_qmmm.nuc_grad_method().kernel()

# LJ
coords_A = test_mol.atom_coords(unit="Angstrom")

E_LJ, grad_LJ = qm_water_lj_energy_gradient(
    coords_A,
    O,
    qm_atom_types
)

grad_LJ_Ha_Bohr = (
    grad_LJ
    / au_to_kJ_conversion
    / 1.889726125
)

print("QM/MM gradient:")
print(grad_qmmm[0])

print("\nLJ gradient:")
print(grad_LJ_Ha_Bohr[0])

print("\nCombined analytic gradient:")
print((grad_qmmm + grad_LJ_Ha_Bohr)[0])

converged SCF energy = -574.305014520737
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0099856768    -0.0002517758    -0.0002165533
1 H     0.0075974117     0.0000311981    -0.0000022025
2 C     0.1137643966     0.0001287242    -0.0268861286
3 Cl     0.0061025534    -0.0007342618    -0.0000192679
4 H    -0.0406633867    -0.0645841297     0.0069044847
5 H    -0.0413168782     0.0647130735     0.0073871398
6 H    -0.0356751281     0.0000087939     0.0142266191
----------------------------------------------
QM/MM gradient:
[-0.00998568 -0.00025178 -0.00021655]

LJ gradient:
[-6.23420617e-07 -2.07806872e-07 -2.07806872e-07]

Combined analytic gradient:
[-0.0099863  -0.00025198 -0.00021676]


In [76]:
optimized_mol, energy_qm, energy_qmmm, E_LJ, result = \
    optimize_qmmm_lj_at_distance(
        test_mol,
        3.0,
        water_x,
        maxsteps=50
    )

coords_opt = optimized_mol.atom_coords(unit="Angstrom")

actual_distance = np.linalg.norm(
    coords_opt[0] - coords_opt[2]
)

print("\nRESULT")
print("Optimization success:", result.success)
print("Message:", result.message)

print("\nTarget C-O distance:")
print(f"{3.0:.8f} Å")

print("\nActual C-O distance:")
print(f"{actual_distance:.8f} Å")

print("\nBare QM energy:")
print(f"{energy_qm:.12f} Hartree")

print("\nQM/MM energy:")
print(f"{energy_qmmm:.12f} Hartree")

print("\nQM/MM - bare QM:")
print(
    (energy_qmmm - energy_qm)
    * au_to_kJ_conversion,
    "kJ/mol"
)

print("\nLJ energy:")
print(f"{E_LJ:.8f} kJ/mol")

print("\nCoordinates:")
print(coords_opt)

converged SCF energy = -574.064478184775
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0017091303     0.0000059620    -0.0003656538
1 H     0.0039182381    -0.0001503650    -0.0000672440
2 C     0.1098146921     0.0000843877    -0.0256860698
3 Cl     0.0063641702    -0.0007127288    -0.0000827654
4 H    -0.0406532444    -0.0633012888     0.0069501058
5 H    -0.0413173082     0.0633877548     0.0074061801
6 H    -0.0366244780     0.0000438489     0.0131913687
----------------------------------------------
converged SCF energy = -574.064478184775
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0017091303     0.0000059620    -0.0003656538
1 H     0.0039182381    -0.0001503650    -0.0000672440
2 C     0.1098146921     0.0000843877    -0.0256860698
3 Cl     0.0063641702    -0.0007127288    -0.0000827654
4 H    -0.0406532444    -0.0633012888     0.0069501058
5 H    -0.041317308

In [77]:
# ============================================================
# STAGE 2: Move one fixed water molecule around a fixed QM system
# ============================================================

coords_A = test_mol.atom_coords(unit="Angstrom")

# Keep the water orientation fixed.
# We will vary only its x-coordinate.
x_positions = np.array([1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5])

results = []

for x in x_positions:

    water_test = water_x.copy()
    water_test[0] = x

    E_total = total_qmmm_lj_energy(
        coords_A,
        water_test
    )

    # Also calculate the two components separately
    O, H1, H2, M = water_from_variables(water_test)

    mm_coords = np.array([H1, H2, M])
    mm_charges = np.array([0.58, 0.58, -1.16])

    mf_qmmm = mm_charge(
        scf.RHF(test_mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf_qmmm.conv_tol = SCF_CONV_TOL
    E_qmmm = mf_qmmm.kernel()

    E_qm = make_scf(test_mol).kernel()

    E_electrostatic = (
        E_qmmm - E_qm
    ) * au_to_kJ_conversion

    E_LJ = qm_water_lj_energy(
        coords_A,
        np.array([O]),
        qm_atom_types
    )

    results.append([
        x,
        E_electrostatic,
        E_LJ,
        E_electrostatic + E_LJ
    ])

print("\nWater position and interaction energies")
print("x (Å)    Electrostatic    LJ          Total")
print("------------------------------------------------")

for row in results:
    print(
        f"{row[0]:5.2f}    "
        f"{row[1]:12.6f}    "
        f"{row[2]:10.6f}    "
        f"{row[3]:10.6f}"
    )

converged SCF energy = -574.30697626062
converged SCF energy = -574.30697626062
converged SCF energy = -574.308632234646
converged SCF energy = -574.305583014022
converged SCF energy = -574.305583014022
converged SCF energy = -574.308632234646
converged SCF energy = -574.305014520737
converged SCF energy = -574.305014520737
converged SCF energy = -574.308632234647
converged SCF energy = -574.305121884932
converged SCF energy = -574.305121884932
converged SCF energy = -574.308632234646
converged SCF energy = -574.3055105401
converged SCF energy = -574.305510540101
converged SCF energy = -574.308632234647
converged SCF energy = -574.305922382178
converged SCF energy = -574.305922382179
converged SCF energy = -574.308632234646
converged SCF energy = -574.306273309922
converged SCF energy = -574.306273309923
converged SCF energy = -574.308632234647

Water position and interaction energies
x (Å)    Electrostatic    LJ          Total
------------------------------------------------
 1.50    

In [78]:
# ============================================================
# STAGE 2B: Numerical derivative of the water-position energy
# ============================================================

def water_total_energy_at_x(x):

    water_test = water_x.copy()
    water_test[0] = x

    return total_qmmm_lj_energy(
        coords_A,
        water_test
    )


delta = 0.001  # Angstrom

test_positions = [2.5, 3.0, 3.5, 4.0]

print("Numerical dE/dx")
print("-----------------------------")

for x in test_positions:

    E_minus = water_total_energy_at_x(x - delta)
    E_plus  = water_total_energy_at_x(x + delta)

    dE_dx = (E_plus - E_minus) / (2 * delta)

    print(
        f"x = {x:.2f} Å   "
        f"dE/dx = {dE_dx:.8f} kJ/mol/Å"
    )


Numerical dE/dx
-----------------------------
converged SCF energy = -574.30501485598
converged SCF energy = -574.305014188273
x = 2.50 Å   dE/dx = -51.92229757 kJ/mol/Å
converged SCF energy = -574.30512126799
converged SCF energy = -574.305122502934
x = 3.00 Å   dE/dx = -45.17063254 kJ/mol/Å
converged SCF energy = -574.3055096885
converged SCF energy = -574.305511391736
x = 3.50 Å   dE/dx = -24.87506659 kJ/mol/Å
converged SCF energy = -574.305921610429
converged SCF energy = -574.30592315366
x = 4.00 Å   dE/dx = -10.59132768 kJ/mol/Å


In [79]:
# ============================================================
# STAGE 2C: Check electrostatic and LJ derivatives separately
# ============================================================

def water_components_at_x(x):

    water_test = water_x.copy()
    water_test[0] = x

    O, H1, H2, M = water_from_variables(water_test)

    # ----------------------------
    # QM/MM electrostatics
    # ----------------------------

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    mf_qmmm = mm_charge(
        scf.RHF(test_mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf_qmmm.conv_tol = SCF_CONV_TOL
    E_qmmm = mf_qmmm.kernel()

    E_qm = make_scf(test_mol).kernel()

    E_electrostatic = (
        E_qmmm - E_qm
    ) * au_to_kJ_conversion

    # ----------------------------
    # LJ
    # ----------------------------

    E_LJ = qm_water_lj_energy(
        coords_A,
        np.array([O]),
        qm_atom_types
    )

    return E_electrostatic, E_LJ


delta = 0.001

test_positions = [2.5, 3.0, 3.5, 4.0]

print("Derivative check")
print()
print(
    "x       dE_elec/dx     dE_LJ/dx       "
    "sum            dE_total/dx"
)
print("-" * 75)

for x in test_positions:

    Eelec_minus, ELJ_minus = water_components_at_x(x - delta)
    Eelec_plus,  ELJ_plus  = water_components_at_x(x + delta)

    dE_elec = (
        Eelec_plus - Eelec_minus
    ) / (2 * delta)

    dE_LJ = (
        ELJ_plus - ELJ_minus
    ) / (2 * delta)

    dE_total = dE_elec + dE_LJ

    # Independent total-energy derivative
    E_total_minus = (
        Eelec_minus + ELJ_minus
    )

    E_total_plus = (
        Eelec_plus + ELJ_plus
    )

    dE_total_direct = (
        E_total_plus - E_total_minus
    ) / (2 * delta)

    print(
        f"{x:3.1f}   "
        f"{dE_elec:14.6f} "
        f"{dE_LJ:14.6f} "
        f"{dE_total:14.6f} "
        f"{dE_total_direct:14.6f}"
    )

Derivative check

x       dE_elec/dx     dE_LJ/dx       sum            dE_total/dx
---------------------------------------------------------------------------
converged SCF energy = -574.30501485598
converged SCF energy = -574.308632234647
converged SCF energy = -574.305014188274
converged SCF energy = -574.308632234646
2.5         0.876531     -52.798829     -51.922299     -51.922299
converged SCF energy = -574.30512126799
converged SCF energy = -574.308632234647
converged SCF energy = -574.305122502934
converged SCF energy = -574.308632234646
3.0        -1.621173     -43.549461     -45.170634     -45.170634
converged SCF energy = -574.3055096885
converged SCF energy = -574.308632234646
converged SCF energy = -574.305511391736
converged SCF energy = -574.308632234646
3.5        -2.235923     -22.639144     -24.875067     -24.875067
converged SCF energy = -574.305921610429
converged SCF energy = -574.308632234646
converged SCF energy = -574.30592315366
converged SCF energy = -574.30863

In [80]:
from scipy.optimize import minimize_scalar

def water_x_energy(x_position):

    x = water_x.copy()
    x[0] = x_position

    return total_qmmm_lj_energy(
        test_mol.atom_coords(unit="Angstrom"),
        x
    )


result = minimize_scalar(
    water_x_energy,
    bounds=(2.5, 4.5),
    method="bounded",
    options={"xatol": 1e-4}
)

print(result)
print()
print("Optimal water x:", result.x)
print("Minimum energy:", result.fun, "kJ/mol")

converged SCF energy = -574.305313350688
converged SCF energy = -574.305710209986
converged SCF energy = -574.305943778122
converged SCF energy = -574.306147943004
converged SCF energy = -574.306072607482
converged SCF energy = -574.306197010037
converged SCF energy = -574.306226600056
converged SCF energy = -574.306244610664
converged SCF energy = -574.30625563703
converged SCF energy = -574.306262411924
converged SCF energy = -574.306266583908
converged SCF energy = -574.306269156571
converged SCF energy = -574.306270744367
converged SCF energy = -574.306271724839
converged SCF energy = -574.306272330485
converged SCF energy = -574.306272704671
converged SCF energy = -574.306272935886
converged SCF energy = -574.306273078767
converged SCF energy = -574.306273167064
converged SCF energy = -574.306273221633
converged SCF energy = -574.306273255356
converged SCF energy = -574.30627327648
 message: Solution found.
 success: True
  status: 0
     fun: -1507841.0340214954
       x: 4.49994

In [81]:
print("Direct energy:")
print(
    total_qmmm_lj_energy(
        test_mol.atom_coords(unit="Angstrom"),
        np.array([4.5, 0.0, 3.0, 0.0, 0.0, 0.0])
    )
)

print()
print("Energy through water_x_energy:")
print(
    water_x_energy(4.5)
)for x in [2.5, 3.0, 3.5, 4.0, 4.5]:

    water = water_x.copy()
    water[0] = x

    E = total_qmmm_lj_energy(
        test_mol.atom_coords(unit="Angstrom"),
        water
    )

    print(f"x = {x:.2f} Å   E = {E:.6f} kJ/mol")

Direct energy:
converged SCF energy = -574.306273309923
-1507841.0342334094

Energy through water_x_energy:
converged SCF energy = -574.306273309922
-1507841.0342334083


In [85]:
for x in [2.5, 3.0, 3.5, 4.0, 4.5, 6, 7, 8]:

    water = water_x.copy()
    water[0] = x

    E = total_qmmm_lj_energy(
        test_mol.atom_coords(unit="Angstrom"),
        water
    )

    print(f"x = {x:.2f} Å   E = {E:.6f} kJ/mol")

converged SCF energy = -574.308632234646
converged SCF energy = -574.305014520737
x = 2.50 Å   E = 60.959196 kJ/mol
converged SCF energy = -574.308632234646
converged SCF energy = -574.305121884932
x = 3.00 Å   E = 35.450554 kJ/mol
converged SCF energy = -574.308632234647
converged SCF energy = -574.305510540101
x = 3.50 Å   E = 17.957178 kJ/mol
converged SCF energy = -574.308632234647
converged SCF energy = -574.305922382179
x = 4.00 Å   E = 9.462324 kJ/mol
converged SCF energy = -574.308632234646
converged SCF energy = -574.306273309923
x = 4.50 Å   E = 6.061461 kJ/mol
converged SCF energy = -574.308632234646
converged SCF energy = -574.306985835747
x = 6.00 Å   E = 3.848688 kJ/mol
converged SCF energy = -574.308632234647
converged SCF energy = -574.307289823393
x = 7.00 Å   E = 3.299221 kJ/mol
converged SCF energy = -574.308632234646
converged SCF energy = -574.307515650024
x = 8.00 Å   E = 2.826349 kJ/mol
